# R2 — Tool-calling agent

A Saffron Shield adjuster assistant that picks the right Unity Catalog tool for the question: claim facts and status come from the data tools (`lookup_claim`, `plan_at`), what the policy says comes from `search_policy` with the section cited. The system prompt lives in `agent/prompts/system_v1.md`, every run is traced with MLflow, and the two client probes (status of CLM-10042, PED waiting period for a corporate group plan) are replayed word for word into `agent/traces/`.

In [0]:
# Config — the single place the catalog and models are named for this notebook
CATALOG = "claims"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"

spark.sql(f"USE CATALOG {CATALOG}")
print(f"catalog={CATALOG}  chat_model={CHAT_MODEL}  embed_model={EMBED_MODEL}")

In [0]:
# System prompt v1 — written to the repo at agent/prompts/system_v1.md and read back unchanged
import os

REPO_DIR = "/Workspace/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1/Gen AI - Paper 5 - Scenario Based Assignment"
SYSTEM_PROMPT_V1 = """You are the Saffron Shield claims assistant for insurance adjusters.
You answer only questions about Saffron Shield claims, plans and policy documents.
For a question about the facts, status or amounts of a claim, call the data tools: lookup_claim for the claim's stored facts, and plan_at when plan terms are needed. Always state those facts exactly as the tools return them - never from memory.
For a question about what the policy says (waiting periods, room rent caps, co-pay, process rules), call search_policy, then answer from the returned sections and cite the policy section by its chunk id, like 03_Waiting_Periods_Group_Plans.md#pre-existing-disease-waiting-period.
Never invent a payout figure: a payout comes only from the compute_payable tool, checked by code. If a question is not about Saffron Shield claims or policies, say so and answer nothing else."""

prompt_path = os.path.join(REPO_DIR, "agent", "prompts", "system_v1.md")
os.makedirs(os.path.dirname(prompt_path), exist_ok=True)
with open(prompt_path, "w") as f:
    f.write(SYSTEM_PROMPT_V1)

with open(prompt_path) as f:
    assert f.read() == SYSTEM_PROMPT_V1, "prompt file round-trip failed"
print(f"wrote {prompt_path} ({len(SYSTEM_PROMPT_V1.split())} words)")

In [0]:
# The four R1 Unity Catalog tools, described for the LLM; executed here against claims.genai
import json

TOOLS = [
  {"type": "function", "function": {
      "name": "lookup_claim",
      "description": "Look up the stored facts of one claim: status, claim type, admission and discharge dates, billed and approved amounts, rejection code, plan at admission, room rent per day and cap, policy tenure, days since policy start, network flag. Returns facts exactly as stored; does not decide a payout.",
      "parameters": {"type": "object", "properties": {"claim_id": {"type": "string", "description": "Claim id, e.g. CLM-10042"}}, "required": ["claim_id"]}}},
  {"type": "function", "function": {
      "name": "plan_at",
      "description": "Return the plan terms (plan code, family, room rent cap, co-pay pct, PED and initial waiting periods, validity dates) a member held on a given date - the policy version valid on that date, never the member's current plan.",
      "parameters": {"type": "object", "properties": {"member_id": {"type": "string", "description": "Member id, e.g. MEM-0437"}, "as_of_date": {"type": "string", "description": "Date as YYYY-MM-DD"}}, "required": ["member_id", "as_of_date"]}}},
  {"type": "function", "function": {
      "name": "search_policy",
      "description": "Search the Saffron Shield policy binder (hybrid keyword plus vector). Returns at most 5 sections with chunk_id, doc_name, heading and text, best first. Use for any question about what the policy says.",
      "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "Natural-language policy question"}}, "required": ["query"]}}},
  {"type": "function", "function": {
      "name": "compute_payable",
      "description": "Deterministically decide a claim's payout by the plan rules. Returns a JSON string with room_rent_over_cap, waiting_period_ok, payable, copay_pct, reason_code and payable_amount. The only source of a payout figure - never compute one yourself.",
      "parameters": {"type": "object", "properties": {
          "plan_code": {"type": "string"},
          "is_pre_existing": {"type": "boolean"},
          "policy_tenure_months": {"type": "integer"},
          "days_since_policy_start": {"type": "integer"},
          "is_network_at_admission": {"type": "boolean"},
          "room_rent_per_day": {"type": "integer"},
          "billed_amount": {"type": "number"}},
        "required": ["plan_code", "is_pre_existing", "policy_tenure_months", "days_since_policy_start", "is_network_at_admission", "room_rent_per_day", "billed_amount"]}}},
]

def _q(s: str) -> str:
    """SQL string literal escaping"""
    return str(s).replace("'", "''")

TOOL_OUTPUTS = {}  # tool name -> list of raw results captured during the latest agent run

def run_tool(name: str, args: dict) -> str:
    if name == "lookup_claim":
        rows = spark.sql(
            f"SELECT * FROM claims.genai.lookup_claim('{_q(args['claim_id'])}')"
        ).collect()
        result = json.dumps([r.asDict() for r in rows], default=str)
    elif name == "plan_at":
        rows = spark.sql(
            f"SELECT * FROM claims.genai.plan_at('{_q(args['member_id'])}', DATE'{_q(args['as_of_date'])}')"
        ).collect()
        result = json.dumps([r.asDict() for r in rows], default=str)
    elif name == "search_policy":
        rows = spark.sql(
            f"SELECT chunk_id, doc_name, heading, text FROM claims.genai.search_policy('{_q(args['query'])}')"
        ).collect()
        result = json.dumps([r.asDict() for r in rows], default=str)
    elif name == "compute_payable":
        row = spark.sql(f"""
          SELECT claims.genai.compute_payable(
            '{_q(args['plan_code'])}', {args['is_pre_existing']}, {args['policy_tenure_months']},
            {args['days_since_policy_start']}, {args['is_network_at_admission']},
            {args['room_rent_per_day']}, {args['billed_amount']}) AS payout
        """).collect()[0]
        result = row.payout
    else:
        result = json.dumps({"error": f"unknown tool {name}"})
    TOOL_OUTPUTS.setdefault(name, []).append(result)
    return result

print(f"{len(TOOLS)} tools registered: {[t['function']['name'] for t in TOOLS]}")

In [0]:
# Chat call (temperature 0) + the agent loop, traced with MLflow on every run
import mlflow
import requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
AUTH = w.config.authenticate()["Authorization"]          # Bearer token, never printed or saved
HOST = "https://" + spark.conf.get("spark.databricks.workspaceUrl")

def chat(messages):
    body = {"messages": messages, "tools": TOOLS, "temperature": 0, "max_tokens": 1024}
    r = requests.post(
        f"{HOST}/serving-endpoints/{CHAT_MODEL}/invocations",
        headers={"Authorization": AUTH, "Content-Type": "application/json"},
        json=body,
    )
    r.raise_for_status()
    return r.json()

@mlflow.trace(name="saffron_shield_agent", span_type="AGENT")
def agent_run(question: str, max_steps: int = 4):
    TOOL_OUTPUTS.clear()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_V1},
        {"role": "user", "content": question},
    ]
    tool_calls_made = []
    for _ in range(max_steps):
        msg = chat(messages)["choices"][0]["message"]
        tcs = msg.get("tool_calls") or []
        if not tcs:
            mlflow.get_current_active_span().set_attributes({
                "question": question,
                "tool_calls": json.dumps(tool_calls_made),
                "final_answer": msg.get("content") or "",
            })
            return {"final_answer": msg.get("content") or "",
                    "tool_calls": tool_calls_made,
                    "trace_id": mlflow.get_active_trace_id()}
        messages.append(msg)
        for tc in tcs:
            name = tc["function"]["name"]
            args = json.loads(tc["function"]["arguments"] or "{}")
            result = run_tool(name, args)
            tool_calls_made.append({"name": name, "args": args})
            messages.append({"role": "tool", "tool_call_id": tc["id"], "content": result})
    raise RuntimeError("agent did not settle within max_steps")

print("agent ready")

In [0]:
# Probe 1 (client replays word for word): claim status
# Checks: only data tools called, lookup_claim for the claim in the question,
# answer states the status exactly as stored in claims.gold.claims_enriched, and the MLflow trace exists.
from mlflow.tracking import MlflowClient

STATUS_QUESTION = "What's the status of claim CLM-10042?"
status_run = agent_run(STATUS_QUESTION)
print(json.dumps(status_run, indent=2))

names = [tc["name"] for tc in status_run["tool_calls"]]
assert all(n in ("lookup_claim", "plan_at") for n in names), f"data tools only, got {names}"
assert any(tc["name"] == "lookup_claim" and tc["args"].get("claim_id") == "CLM-10042"
           for tc in status_run["tool_calls"]), "lookup_claim('CLM-10042') was not called"

stored_status = spark.sql(
    "SELECT status FROM claims.gold.claims_enriched WHERE claim_id = 'CLM-10042'"
).first()["status"]
assert stored_status.lower() in status_run["final_answer"].lower(), \
    f"answer must state the stored status '{stored_status}'"

trace = MlflowClient().get_trace(status_run["trace_id"])
assert trace is not None, "trace not found in MLflow"
print(f"OK: tools={names}  stored status='{stored_status}'  trace {status_run['trace_id']} exists")

In [0]:
# Probe 2 (client replays word for word): policy question
# Checks: only search_policy called, the grounding sections include a 03_Waiting_Periods_Group_Plans.md chunk
# and no 02_Waiting_Periods_Individual_Plans.md chunk is relied on, and the MLflow trace exists.
import re

POLICY_QUESTION = "What is the pre-existing disease (PED) waiting period for a corporate group plan?"
policy_run = agent_run(POLICY_QUESTION)
print(json.dumps(policy_run, indent=2))

names = [tc["name"] for tc in policy_run["tool_calls"]]
assert set(names) == {"search_policy"}, f"only search_policy may be called, got {names}"

# chunks the agent relied on = what search_policy returned during the run
grounded = [row["chunk_id"]
            for result in TOOL_OUTPUTS.get("search_policy", [])
            for row in json.loads(result)]

cited = sorted(set(re.findall(r"\d{2}_[A-Za-z_]+\.md#[a-z0-9\-]+", policy_run["final_answer"])) or set(grounded))
assert any(c.startswith("03_Waiting_Periods_Group_Plans.md#") for c in cited), f"must cite a 03_ chunk, got {cited}"
assert not any(c.startswith("02_Waiting_Periods_Individual_Plans.md#") for c in cited), f"must not cite an 02_ chunk, got {cited}"

trace = MlflowClient().get_trace(policy_run["trace_id"])
assert trace is not None, "trace not found in MLflow"
print(f"OK: tools={names}  cited={cited}  trace {policy_run['trace_id']} exists")

In [0]:
# Evidence — agent/traces/r2_status_probe.json and r2_policy_probe.json
# Each: trace_id (an MLflow trace that exists), tool_calls (name + args), final_answer, cited_chunk_ids (bare chunk ids).
status_evidence = {
    "trace_id": status_run["trace_id"],
    "tool_calls": status_run["tool_calls"],
    "final_answer": status_run["final_answer"],
    "cited_chunk_ids": [],
}
policy_evidence = {
    "trace_id": policy_run["trace_id"],
    "tool_calls": policy_run["tool_calls"],
    "final_answer": policy_run["final_answer"],
    "cited_chunk_ids": [c for c in cited if c.startswith("03_Waiting_Periods_Group_Plans.md#")],
}

traces_dir = os.path.join(REPO_DIR, "agent", "traces")
os.makedirs(traces_dir, exist_ok=True)
for name, payload in [("r2_status_probe.json", status_evidence), ("r2_policy_probe.json", policy_evidence)]:
    path = os.path.join(traces_dir, name)
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    with open(path) as f:  # round-trip check
        assert json.load(f) == payload
    print(f"wrote {path}")

# both traces still exist in MLflow
for tid in (status_evidence["trace_id"], policy_evidence["trace_id"]):
    assert MlflowClient().get_trace(tid) is not None, f"trace {tid} missing"
print("both traces verified in MLflow")